# 1. Librairies importation

In [3]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
requirements_path = project_root / "requirements.txt"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%pip install -r "{requirements_path}"
%pip install deepchem

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 2. Adapting our columns


In [4]:
import pandas as pd

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_reg = pd.read_csv(processed_dir / "solubility_clean.csv")

# 3. Bemis-Murcko scaffolds to separate molecules with the same scaffold in training and test

In [16]:
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    scaffold = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
    return scaffold if scaffold else smiles


smiles_col = "Canonical_smiles"
df_reg["scaffold"] = df_reg[smiles_col].apply(get_scaffold)

invalid_count = df_reg["scaffold"].isna().sum()
print("Invalid SMILES:", invalid_count)

# Remove molecules with invalid structures.
df_reg = df_reg.dropna(subset=["scaffold"]).copy()

print("Valid molecules:", len(df_reg))
print("Unique scaffolds:", df_reg["scaffold"].nunique())


[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:06] WARNING: not removing hydrogen atom without neighbors
[11:44:07] WARNING: not removing hydrogen atom without neighbors
[11:44:07] WARNING: not removing hydrogen atom without neighbors
[11:44:07] WARNING: not removing hydrogen atom without neighbors
[11:44:07] WARNING: not r

Invalid SMILES: 0
Valid molecules: 9982
Unique scaffolds: 4996


# 4. Test-split 

In [17]:
# Size of each scaffold group
scaffold_sizes = df_reg["scaffold"].value_counts()

print(scaffold_sizes.describe())
print("Largest scaffold group:", scaffold_sizes.max())

# Splitting our molecules by scafold 
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df_reg, groups=df_reg["scaffold"])
)

train_df = df_reg.iloc[train_idx].copy()
test_df = df_reg.iloc[test_idx].copy()

print("Training set:", len(train_df), "molecules")
print("Test set:", len(test_df), "molecules")
print("Test proportion:", round(len(test_df) / len(df_reg) * 100, 2), "%")


count    4996.000000
mean        1.997998
std        25.393243
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max      1753.000000
Name: count, dtype: float64
Largest scaffold group: 1753
Training set: 8354 molecules
Test set: 1628 molecules
Test proportion: 16.31 %


# 5. Freeze the split from test-split 

In [18]:
df_reg["split"] = "train"
df_reg.loc[test_df.index, "split"] = "test"

df_reg.to_csv(processed_dir / "saved_scaffold_split_solubility.csv", index=False)

print(df_reg["split"].value_counts())
print("Split assignments saved showing which columns will be used for training and which will be used for testing.")

split
train    8354
test     1628
Name: count, dtype: int64
Split assignments saved showing which columns will be used for training and which will be used for testing.


# 6. Checking if there is any leakage between training and testing data 

In [7]:
df = pd.read_csv(project_root / "data" / "processed" / "saved_scaffold_split_solubility.csv")
train_df = df[df["split"] == "train"]
test_df = df[df["split"] == "test"]

assert set(train_df["scaffold"]).isdisjoint(set(test_df["scaffold"]))
print(train_df["Solubility"].describe())
print(test_df["Solubility"].describe())

count    8354.000000
mean       -2.896363
std         2.369664
min       -12.950000
25%        -4.313364
50%        -2.600150
75%        -1.223119
max         2.137682
Name: Solubility, dtype: float64
count    1628.000000
mean       -2.856788
std         2.360840
min       -13.171900
25%        -4.362500
50%        -2.678983
75%        -1.085275
max         1.700916
Name: Solubility, dtype: float64


# 7. Using the descriptors

In [19]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

def featurize(smiles_series):
    rows = []
    for smi in smiles_series:
        mol = Chem.MolFromSmiles(smi)
        row = []
        for name, func in Descriptors.descList:
            try:
                row.append(func(mol))
            except Exception:
                row.append(np.nan)
        rows.append(row)
    return np.array(rows, dtype=float)

X_train_all = featurize(train_df["Canonical_smiles"].values)
X_test_all = featurize(test_df["Canonical_smiles"].values)

y_train = train_df["Solubility"].values
y_test = test_df["Solubility"].values
groups_train = train_df["scaffold"].values

print("Descriptors computed:", X_train_all.shape[1])

# remove the descriptors with invalid values (same idea as in the course)
good_columns = np.isfinite(X_train_all).all(axis=0) & np.isfinite(X_test_all).all(axis=0)
X_train = X_train_all[:, good_columns]
X_test = X_test_all[:, good_columns]

print("Descriptors kept:", X_train.shape[1])
print("Train:", X_train.shape, "Test:", X_test.shape)

[11:45:04] WARNING: not removing hydrogen atom without neighbors
[11:45:04] WARNING: not removing hydrogen atom without neighbors
[11:45:04] WARNING: not removing hydrogen atom without neighbors
[11:45:07] WARNING: not removing hydrogen atom without neighbors
[11:45:07] WARNING: not removing hydrogen atom without neighbors
[11:45:07] WARNING: not removing hydrogen atom without neighbors
[11:45:07] WARNING: not removing hydrogen atom without neighbors
[11:45:07] WARNING: not removing hydrogen atom without neighbors
[11:45:07] WARNING: not removing hydrogen atom without neighbors
[11:45:15] WARNING: not removing hydrogen atom without neighbors
[11:45:15] WARNING: not removing hydrogen atom without neighbors
[11:45:15] WARNING: not removing hydrogen atom without neighbors
[11:45:21] WARNING: not removing hydrogen atom without neighbors
[11:45:21] WARNING: not removing hydrogen atom without neighbors
[11:45:21] WARNING: not removing hydrogen atom without neighbors
[11:45:24] WARNING: not r

Descriptors computed: 210
Descriptors kept: 198
Train: (8354, 198) Test: (1628, 198)


# 7.1 : Saving results from #7 to avoird running it everytime (6min)

In [20]:
np.save(project_root / "data" / "processed" / "Solubility_X_train_all.npy", X_train_all)
np.save(project_root / "data" / "processed" / "Solubility_X_test_all.npy", X_test_all)


# 8. Cross Valditation (on training data only) 

In [21]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler

# remove the descriptors that have the same value for all molecules
selector = VarianceThreshold(threshold=0.0)
X_train_s = selector.fit_transform(X_train)
X_test_s = selector.transform(X_test)

# normalize: the scaler learns only from the training set
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train_s)
X_test_s = scaler.transform(X_test_s)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

/Users/erin.so/miniforge3/envs/chemiinfo/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1746: RuntimeWarning: overflow encountered in multiply
  sqr = np.multiply(arr, arr, out=arr, where=where)


Train: (8354, 195) Test: (1628, 195)


In [13]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 20, 30]
}

# 5-folds cross validation (we keep the molecules with the same scaffold together)
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=0),
    param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="neg_root_mean_squared_error"
)

# only training data 
grid_search.fit(X_train_s, y_train, groups=groups_train)

print("Best parameters:", grid_search.best_params_)
print("Best RMSE (validation):", round(-grid_search.best_score_, 3))

Best parameters: {'max_depth': 30, 'n_estimators': 100}
Best RMSE (validation): 1.107


# 8.1 To avoid running [13] again (around 18 min)

In [14]:
import json
import pandas as pd

processed_dir = project_root / "data" / "processed"

# 1. save the best parameters
with open(processed_dir / "rf_best_params.json", "w") as f:
    json.dump(grid_search.best_params_, f)

# 2. save the results of all the combinations tested
pd.DataFrame(grid_search.cv_results_).to_csv(
    processed_dir / "rf_grid_search_results.csv", index=False
)

print("Saved in:", processed_dir)
print("Best parameters:", grid_search.best_params_)
print("Best RMSE (validation):", round(-grid_search.best_score_, 3))

Saved in: /Users/erin.so/Documents/Cheminformatics/Data-Project-for-Cheminformatics-class/data/processed
Best parameters: {'max_depth': 30, 'n_estimators': 100}
Best RMSE (validation): 1.107


# 9. Train and evaluate the model 
9.1 Reload the saved data and the best hyperparameters

In [10]:
import json
import numpy as np
import pandas as pd


from pathlib import Path
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "data" / "processed").exists()), cwd)
processed_dir = project_root / "data" / "processed"

if "X_train_s" in globals() and "groups_train" in globals():
    print("X_train_s, X_test_s, y_train, y_test and groups_train are already defined.")
else:
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.preprocessing import MinMaxScaler

    df = pd.read_csv(processed_dir / "saved_scaffold_split_solubility.csv")
    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    y_train = train_df["Solubility"].values
    y_test = test_df["Solubility"].values
    groups_train = train_df["scaffold"].values

    X_train_all = np.load(processed_dir / "Solubility_X_train_all.npy")
    X_test_all = np.load(processed_dir / "Solubility_X_test_all.npy")

    # same preprocessing as in sections 7 and 8
    good_columns = np.isfinite(X_train_all).all(axis=0) & np.isfinite(X_test_all).all(axis=0)
    X_train = X_train_all[:, good_columns]
    X_test = X_test_all[:, good_columns]

    selector = VarianceThreshold(threshold=0.0)
    X_train_s = selector.fit_transform(X_train)
    X_test_s = selector.transform(X_test)

    scaler = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train_s)
    X_test_s = scaler.transform(X_test_s)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

# the best hyperparameters found by the grid search (saved in section 8.1)
with open(processed_dir / "rf_best_params.json") as f:
    best_params = json.load(f)
print("Best parameters:", best_params)

# the results of all the combinations tested (RMSE = - neg_root_mean_squared_error)
grid_results = pd.read_csv(processed_dir / "rf_grid_search_results.csv")
grid_results["RMSE validation"] = -grid_results["mean_test_score"]
grid_results[["param_max_depth", "param_n_estimators", "RMSE validation",
              "std_test_score", "rank_test_score"]].sort_values("rank_test_score").round(3)

Train: (8354, 195) Test: (1628, 195)
Best parameters: {'max_depth': 30, 'n_estimators': 100}


/Users/erin.so/miniforge3/envs/chemiinfo/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1746: RuntimeWarning: overflow encountered in multiply
  sqr = np.multiply(arr, arr, out=arr, where=where)


,param_max_depth,param_n_estimators,RMSE validation,std_test_score,rank_test_score
7,30,100,1.107,0.115,1
5,20,100,1.108,0.116,2
6,30,50,1.114,0.113,3
4,20,50,1.115,0.113,4
3,10,100,1.137,0.123,5
2,10,50,1.140,0.121,6
1,5,100,1.250,0.153,7
0,5,50,1.252,0.152,8


Conclusion : The best model is the one with max_depth=30 and n_estimators=100, with a validation RMSE of 1.107. 

The depth changes the results the most: with max_depth=5 the RMSE is around 1.25, and it gets better until depth 20. After that it doesn’t really change (1.108 for depth 20 and 1.107 for depth 30). 

The number of trees matters very little, 100 trees is just a bit better than 50.

The std between the folds is about 0.115, so it is bigger than the differences between the best models. So I think depth 20 and depth 30 are basically the same, but we keep the best one from the grid search. We will see in section 11 if the test RMSE is close to 1.107.


9.2 The function train_test_model trains a model on the training set, predicts the train and test sets, prints the RMSE of both and returns them, so that all the models are evaluated in the same way.

In [11]:
from sklearn.metrics import mean_squared_error

def train_test_model(model, X_train, y_train, X_test, y_test):
    """
    Function that trains a model, and tests it.
    Inputs: sklearn model, train_data, test_data
    """
    # Train model
    model.fit(X_train, y_train)

    # Calculate RMSE on training and test
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    model_train_mse = mean_squared_error(y_train, y_pred_train)
    model_test_mse = mean_squared_error(y_test, y_pred_test)
    model_train_rmse = model_train_mse ** 0.5
    model_test_rmse = model_test_mse ** 0.5
    print(f"RMSE on train set: {model_train_rmse:.3f}, and test set: {model_test_rmse:.3f}.\n")

    return model_train_rmse, model_test_rmse

# 10. Create models and finding the best one 
A baseline that always predicts the mean of the training set gives the minimum RMSE to beat.

In [12]:
# baseline: always predicts the mean of the training targets
from sklearn.dummy import DummyRegressor
dummy_reg = DummyRegressor(strategy="mean")

# random forest regressor, and the default criterion is mean squared error (MSE)
from sklearn.ensemble import RandomForestRegressor
ranf_reg = RandomForestRegressor(n_estimators=10, random_state=0)  # using 10 trees and seed=0

# XGBoost regressor
from xgboost import XGBRegressor
xgb_reg = XGBRegressor(n_estimators=10, random_state=0)  # using 10 trees and seed=0

# SVM regressor with default parameters
from sklearn.svm import SVR
svm_reg = SVR()

In [13]:
results = {}

# Train and test the baseline
print("Evaluating Baseline (mean).")
results["Baseline (mean)"] = train_test_model(dummy_reg, X_train_s, y_train, X_test_s, y_test)

# Train and test the random forest model
print("Evaluating Random Forest Model.")
results["Random Forest (default)"] = train_test_model(ranf_reg, X_train_s, y_train, X_test_s, y_test)

# Train and test XGBoost model
print("Evaluating XGBoost model.")
results["XGBoost (default)"] = train_test_model(xgb_reg, X_train_s, y_train, X_test_s, y_test)

# Train and test SVM model
print("Evaluating SVM model.")
results["SVR (default)"] = train_test_model(svm_reg, X_train_s, y_train, X_test_s, y_test)

Evaluating Baseline (mean).
RMSE on train set: 2.370, and test set: 2.360.

Evaluating Random Forest Model.
RMSE on train set: 0.458, and test set: 1.196.

Evaluating XGBoost model.
RMSE on train set: 0.889, and test set: 1.174.

Evaluating SVM model.
RMSE on train set: 1.177, and test set: 1.330.



# 10.1 Which model is better in this case ?

All the models are much better than the baseline (2.360), so they really learned something. On the test set, XGBoost is the best (1.174), then the Random Forest (1.196) and then the SVM (1.330). But the difference between XGBoost and the Random Forest is small (0.02), so I would not say that one is clearly better, especially because the RF only has 10 trees here.

The RF has a very low train RMSE (0.458) and a much higher test RMSE (1.196), so it overfits a lot. XGBoost overfits less (0.889 vs 1.174). The SVM has almost the same RMSE on train and test, but it is the worst, so maybe it is not flexible enough with the default parameters.

For now XGBoost is the best, but all these models use default parameters. 

So we will see in section 11 if the Random Forest gets better with the best hyperparameters.

# 11. Hyperparameter optimization
The best hyperparameters were found with GridSearchCV on the training data only (grid_search.fit(X_train_s, y_train, groups=groups_train)), using GroupKFold so that molecules with the same scaffold stay together. The test set was not used. We now re-train the random forest with these parameters and evaluate it on the test set.

In [14]:
# re-train a model using best hyperparameters
rf_gs = RandomForestRegressor(**best_params, random_state=0)

print('Random forests performance after hyperparameter optimization:')
results["Random Forest (GridSearch)"] = train_test_model(rf_gs, X_train_s, y_train, X_test_s, y_test)

Random forests performance after hyperparameter optimization:
RMSE on train set: 0.389, and test set: 1.105.



**Results analysis**

The test RMSE is 1.105, much better than the baseline (2.360). 

The train RMSE is lower (0.389), so there is still some overfitting. 

But the test RMSE is almost the same as the validation RMSE from the grid search (1.107), so we can say the model generalizes well to new scaffolds.

**Comparison of `ranf_reg` and `rf_gs`**

Both are Random Forests. `ranf_reg` is the default one from section 10 (10 trees, no hyperparameter tuning), and `rf_gs` is the one with the best hyperparameters from the grid search (`max_depth=30`, `n_estimators=100`).

`rf_gs` is better: its test RMSE is 1.105 and the default RF has 1.196, and both are way better than the baseline (2.360). But both overfit a lot, because the train RMSE is much lower than the test one (0.389 vs 1.105 for `rf_gs`). So the optimization didn't really fix the overfitting, it just made the test error smaller.

The test RMSE (1.105) is almost the same as the validation RMSE from the grid search (1.107), so I think the model generalizes well to new scaffolds.

# 12. Summary of the results and written conclusion 

In [15]:
summary = pd.DataFrame(results, index=["RMSE train", "RMSE test"]).T
summary["Gap (test - train)"] = summary["RMSE test"] - summary["RMSE train"]
summary.sort_values("RMSE test").round(3)

,RMSE train,RMSE test,Gap (test - train)
Random Forest (GridSearch),0.389,1.105,0.716
XGBoost (default),0.889,1.174,0.285
Random Forest (default),0.458,1.196,0.738
SVR (default),1.177,1.330,0.153
Baseline (mean),2.370,2.360,-0.009
